In [2]:
import numpy as np # linear algebra
# import pandas as pd # data processing

import os

In [3]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [4]:
# !cd /kaggle/working

In [5]:
tracking_annotation_path = os.path.join("/kaggle/input/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation")
root_working_path = "/kaggle/working"

In [6]:
def get_videos_dirs():
    train_dirs = ["1", "3", "6", "7", "10", "13", "15", "16", "18", "22", "23", "31", "32", "36", "38", "39", "40",
                 "41", "42", "48", "50", "52", "53", "54"]
    train_dirs.sort()

    val_dirs = ["0", "2", "8", "12", "17", "19", "24", "26", "27", "28", "30", "33", "46", "49", "51"]
    val_dirs.sort()

    test_dirs = ['4', '5', '9', '11', '14', '20', '21', '25', '29', '34', '35', '37', '43', '44', '45', '47']
    test_dirs.sort()

    return train_dirs, val_dirs, test_dirs

### Group and Person Category

In [7]:
from logging import root
import os
import pickle

# root_videos_path = '/kaggle/input/volleyball/volleyball_/videos'
# root_output_path = '/kaggle/working'
# output_annot_path = "/kaggle/working/volleyball-baseline-annotations"

# if not os.path.exists(output_annot_path):
#     os.makedirs(output_annot_path)

def prep_categories():
    group_categories = {
        'l-pass': 0,
        'r-pass': 1,
        'l-spike': 2,
        'r_spike': 3,
        'l_set': 4,
        'r_set': 5,
        'l_winpoint': 6,
        'r_winpoint': 7
    }

    person_categories = {
        'standing': 0,
        'setting': 1,
        'waiting': 2,
        'moving': 3,
        'falling': 4,
        'spiking': 5,
        'jumping': 6,
        'digging': 7,
        'blocking': 8
    }
    
    return group_categories, person_categories


### Box Info (Comparable)

In [8]:
# Box info for every single player
class BoxInfo:
    def __init__(self, line):
        words = line.split()

        self.category = words.pop()
        words = [int(word) for word in words]

        player_id, x1, y1, x2, y2, frame_id, lost, grouping, generated = words
        self.player_id = player_id
        self.box = [x1, y1, x2, y2]
        self.frame_id = frame_id
        self.lost = lost
        self.grouping = grouping
        self.generated = generated

    def get_box_info(self):
        return {'frame_id': self.frame_id,
                'box': self.box,
                'category': self.category
                }

### Tracking annotation for every single player within 20 frame per clip for the 12 players  (Comparable)

In [9]:

# tracking_annot_path = 'volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation'
def get_players_boxes(tracking_annot_path):
    # load tracking annotations for one clip
    with open(tracking_annot_path, 'r') as file:
        player_boxes = {idx:[] for idx in range(12)}

        for line in file:
            box_info = BoxInfo(line)
            player_id, box, lost, category = box_info.player_id, box_info.box, box_info.lost, box_info.category
            box_info_dct = box_info.get_box_info()

            # if number of players is more than 12 by mistake stop on player number 12 and ignore others
            if box_info.player_id > 11:
                continue

            player_boxes[box_info.player_id].append(box_info_dct)

        frame_boxes_dct = {}
        for player_id, boxes_info in player_boxes.items():
            # for baseline 3, I need just 4 frames before and 4 after the target [5:13] from 6 to 14 ignoring zero indexing
            boxes_info = boxes_info[5:-6]
            # player_boxes[player_id] = boxes_info[:-6]

            for box_info in boxes_info:
                # print(box_info)
                player_box = {}
                box, category = box_info['box'], box_info['category']
                player_box = {
                    'player_id': player_id,
                    'box': box,
                    'category': category
                }
                # print(box_info)

                if box_info['frame_id'] not in frame_boxes_dct:
                    # print(box_info.frame_id)
                    frame_boxes_dct[box_info['frame_id']] = []

                frame_boxes_dct[box_info['frame_id']].append(player_box)
        frame_boxes_lst_dct = {}
        for frame_id, boxes_lst in frame_boxes_dct.items():
            player_id = []
            box = []
            category = []

            for boxes in boxes_lst:
                player_id.append(boxes['player_id'])
                box.append(boxes['box'])
                category.append(boxes['category'])

            frame_boxes_lst_dct[frame_id] = [player_id, box, category]
            # print(frame_boxes_lst_dct[frame_id])

        '''
        It returns 9 frames per clip, each frame has 12 player boxes.
        RETURN: --> { 'frame_id': [box1, box2, ......., box12] }
        '''
        # print(frame_boxes_lst_dct)
        return frame_boxes_lst_dct

In [10]:
tracking_annot_path = '/kaggle/input/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation'
def load_tracking_annots(dirs_list, tracking_annot_path: str):
    print(f'Loading tracking annotations for {len(dirs_list)} videos from {tracking_annot_path}')
    clip_track_annots = []
    person_activity_encode = prep_categories()[1]

    for vid in dirs_list:
        if not os.path.exists(os.path.join(tracking_annot_path, vid)):
            continue

        tracking_annots_vid_dir = os.listdir(os.path.join(tracking_annot_path, vid))
        tracking_annots_vid_dir.sort()
        # print(clips_dir)
        # print(tracking_annots_dir)

        for clip in tracking_annots_vid_dir:
            if not os.path.isdir(os.path.join(tracking_annot_path, vid, clip)):
                continue

            players_boxes = get_players_boxes(os.path.join(tracking_annot_path, vid, clip, f'{clip}.txt'))

            for frame_id, boxes in players_boxes.items():
                category = []
                for cat in boxes[2]:
                    category.append(person_activity_encode[cat])

                clip_track_annots_players_dct = {
                    'video': vid,
                    'clip': clip,
                    'frame_id': frame_id,
                    'players_id': boxes[0],
                    'boxes': boxes[1],
                    'category': category
                }
                # print(clip_track_annots_players_dct)
                clip_track_annots.append(clip_track_annots_players_dct)
            # print(clip_track_annots)
    return clip_track_annots

In [11]:
from PIL import Image

def get_cropped_images(videos_path, annot_lst):
    frame_boxes_lst = []
    for annot_dct in annot_lst:
        vid = annot_dct['video']
        clip = annot_dct['clip']
        frame_id = annot_dct['frame_id']
        image_path = os.path.join(videos_path, vid, clip, f'{frame_id}.jpg')
        image = Image.open(image_path).convert('RGB')

        players_id = []
        cropped_boxes = []
        categories = []

        for player_id in annot_dct['players_id']:
            players_id.append(player_id)

        for box in annot_dct['boxes']:
            cropped_box = image.crop(box)
            cropped_boxes.append(cropped_box)

            # cv2.imshow("Image", np.array(cropped_box))
            # cv2.waitKey(0)
            # cv2.destroyAllWindows()


        for cat in annot_dct['category']:
            categories.append(cat)

        frame_boxes_lst.append([players_id, cropped_boxes, categories])

    return frame_boxes_lst

In [12]:
import pathlib

root_path = 'kaggle/input'

training_output_path = "/kaggle/working/training-outputs"
if not os.path.exists(training_output_path):
    os.makedirs(training_output_path)


def prepare_dataset():
    print("Preparing dataset...")
    # get train, val and test folders in a sorted way

    # root_path = pathlib.Path.cwd().parents[2]
    videos_path = '/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos'

    output_annot_path = os.path.join(training_output_path)
    if not os.path.exists(output_annot_path):
        os.makedirs(output_annot_path)

    # tracking_annot_path = os.path.join(root_path,
    #                                         'volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation')

    
    tracking_annot_path = '/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation'


    # print(root_path)
    train, val, test = get_videos_dirs()

    train_annots = load_tracking_annots(train, tracking_annot_path)
    # train_crops = get_cropped_images(videos_path, train_annots)

    # print('the length of annot : ' + str(len(train_annots)))
    # print(type(train_annots[0]))
    # print((train_annots[0]))
    # print(type(train_annots[0]['box']))

    val_annots = load_tracking_annots(val, tracking_annot_path)
    # val_crops = get_cropped_images(videos_path, val_annots)

    test_annots = load_tracking_annots(test, tracking_annot_path)
    # test_crops = get_cropped_images(videos_path, test_annots)


    with open(os.path.join(output_annot_path, 'train_players_annots.pickle'), 'wb') as tr_file:
        pickle.dump(train_annots, tr_file, pickle.HIGHEST_PROTOCOL)

    with open(os.path.join(training_output_path, 'val_players_annots.pickle'), 'wb') as vl_file:
        pickle.dump(val_annots, vl_file, pickle.HIGHEST_PROTOCOL)

    with open(os.path.join(training_output_path, 'test_players_annots.pickle'), 'wb') as ts_file:
        pickle.dump(test_annots, ts_file, pickle.HIGHEST_PROTOCOL)


it took 15 minutes to prepare the train/val/test data pickle files

In [13]:
prepare_dataset()

Preparing dataset...
Loading tracking annotations for 24 videos from /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation
Loading tracking annotations for 15 videos from /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation
Loading tracking annotations for 16 videos from /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation


In [ ]:
import os
import pickle

import cv2
import numpy as np

import torchvision.transforms as transforms

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random


class VolleyBallPersonDataLevel(Dataset):

    def __init__(self, root_videos_path, data_list, preprocess=None, shuffle=False):
        # super().__init__(self)
        self.videos_path = root_videos_path  # ✅ FIXED: was videos_path (undefined)
        self.data_list = data_list
        # self.preprocess = preprocess if preprocess is not None else get_default_preprocessor()
        self._shuffle(shuffle)

    # At first, we train on person activity so we will feed the boxes alone with no group activity

    def __getitem__(self, idx):
        """
        Returns a single sample: (processed_crops, labels)
        
        Returns:
            - processed_crops: tensor [12, 3, 224, 224] in float16 (length 12, padded if needed)
            - processed_labels: tensor [12] with -1 for padding (int64/long)
        """
        vid, clip, frame = self.data_list[idx]['video'], self.data_list[idx]['clip'], self.data_list[idx]['frame_id']

        image_path = os.path.join(self.videos_path, vid, clip, f'{frame}.jpg')
        image = Image.open(image_path).convert('RGB')

        boxes, player_category = self.data_list[idx]['boxes'], self.data_list[idx]['category']

        processed_crops = []
        processed_labels = []

        # ✅ FIXED: Process each box correctly
        for box, label in zip(boxes, player_category):
            cropped_box = image.crop(box)
            processed_crop = preprocessor(cropped_box)
            processed_crops.append(processed_crop)
            processed_labels.append(label)

        # Pad to exactly 12 players with zero tensors and -1 labels
        while len(processed_crops) < 12:
            zero_crop = torch.zeros((3, 224, 224), dtype=torch.float16)  # ✅ CHANGED: float16 saves 50% memory
            processed_crops.append(zero_crop)
            processed_labels.append(-1)

        # Ensure exactly 12 players (truncate if more)
        processed_crops = torch.stack(processed_crops[:12]).to(torch.float16)  # ✅ Convert to float16
        processed_labels = torch.tensor(processed_labels[:12], dtype=torch.long)  # ✅ Already correct: torch.long

        # ✅ FIXED: Close image to free memory (important with num_workers)
        image.close()

        return processed_crops, processed_labels

    def __len__(self):
        return len(self.data_list)

    def _shuffle(self, shuffle):
        if shuffle:
            random.shuffle(self.data_list)

def collate_fn(batch):
    """
    Custom collate function for DataLoader.
    
    Input batch: list of tuples (processed_crops, processed_labels)
        - Each tuple contains tensors of shape [12, 3, 224, 224] in float16
    
    Output:
        - images: Tensor [B, 12, 3, 224, 224] in float16
        - labels: Tensor [B, 12] with dtype=long
    """
    crops, labels = zip(*batch)
    images = torch.stack(crops)
    labels = torch.stack(labels)

    return images, labels

def preprocessor(image):
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])(image).to(torch.float16)  # ✅ Convert output to float16


In [15]:
from IPython.display import FileLink
from IPython.display import display

# annot_dir = "volleyball-baseline-annotations"

annot_train_pkl = "/kaggle/working/training-outputs/train_players_annots.pickle"
annot_val_pkl = "/kaggle/working/training-outputs/val_players_annots.pickle"
# annot_test_pkl = "/kaggle/working/training-outputs/test_players_annots.pickle"


# path = f'{annot_dir}/b3-test-annot.pickle'
# if not os.path.exists(annot_dir):
#     os.makedirs(annot_dir)
   
# display(FileLink(annot_train_pkl, result_html_prefix="click here to download: "))
# display(FileLink(annot_val_pkl, result_html_prefix="click here to download: "))
# display(FileLink(annot_test_pkl, result_html_prefix="click here to download: "))

In [16]:
# train_pkl_path = "/kaggle/working/volleyball-baseline-annotations/b3-train-annot.pickle"
# val_pkl_path = "/kaggle/working/volleyball-baseline-annotations/b3-val-annot.pickle"
# test_pkl_path = "/kaggle/working/volleyball-baseline-annotations/b3-test-annot.pickle"



In [ ]:
import math
import gc  # ✅ ADDED: For garbage collection
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from torch import nn
import torchvision.models as models
import torch.optim as optim


class PersonLevelModel(nn.Module):
    def __init__(self, num_classes):
        super(PersonLevelModel, self).__init__()
        self.backbone_model = None
        self.classifier = None
        self.num_classes = num_classes

        self.optimizer_config = None
        self.criterion = None
        self.accuracy = None
        self.save_interval = None

        self._prepare_model()

    def _prepare_model(self):
        model = models.resnet50(pretrained=True)
        model = nn.Sequential(*(list(model.children())[:-1]))

        fc_layers = nn.Sequential(
            nn.Dropout(0.5, inplace=False),
            nn.Linear(2048, self.num_classes)
            # nn.BatchNorm1d(18, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            # nn.ReLU(inplace=True),
            # nn.Dropout(0.5, inplace=False),
            # nn.Linear(18, self.num_classes)
        )
        self.backbone_model = model
        self.classifier = fc_layers

    def model_summary(self):
        print(f'backbone model')
        print(self.backbone_model)
        
        print(f'classifier')
        print(self.classifier)

    # def _optimizers(self, optim):
    #     optims = dict(
    #         Adam=torch.optim.Adam([{'params': self.backbone_model.parameters()},
    #                                {'params': self.classifier.parameters()}],
    #                               lr=optim['lr'],
    #                               weight_decay=optim['weight_decay'])
    #     )
    #     return optims[optim['optimizer']]

    def set_metrics(self, optimizer, criterion, accuracy, save_interval=5):
        self.optimizer_config = optimizer
        self.criterion = criterion
        self.accuracy = accuracy
        self.save_interval = save_interval

    def train_model(self, trainLoader, backbone_model, classifier, optimizer, device):
        backbone_model.train()
        classifier.train()

        criterion = self.criterion
        train_loss_per_batch = 0
        total_correct_predictions = 0
        num_of_steps = len(trainLoader.dataset) / len(trainLoader)

        for batch_idx, (data, target) in enumerate(trainLoader):
            # The input shape is x: [B, 12, 3, 224, 224]
            B, P, C, H, W = data.shape

            data = data.view(-1, C, H, W)
            target = target.view(-1)

            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()

            # with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = backbone_model(data)
            output = output.view(output.size(0), -1)
            output = classifier(output)
            loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            train_loss_per_batch += loss.item() * data.size(0)

            prediction = torch.argmax(output, dim=1)
            correct_predictions = sum(pred == tar for pred, tar in zip(prediction, target)).item()

            total_correct_predictions += correct_predictions

            # ✅ ADDED: Clear cache after each batch to prevent OOM
            del data, target, output, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        total_loss = train_loss_per_batch / len(trainLoader.dataset)
        total_accuracy = total_correct_predictions * 100 / len(trainLoader.dataset)

        return backbone_model, classifier, optimizer, total_loss, total_accuracy

    def eval_model(self, valLoader, backbone_model, classifier, device):

        backbone_model = backbone_model
        classifier = classifier

        val_loss_per_batch = 0
        total_correct_predictions = 0
        backbone_model.eval(), classifier.eval()

        with torch.no_grad():
            
            for batch_idx, (data, target) in enumerate(valLoader):
                # The input shape is x: [B, 12, 3, 224, 224]
                B, P, C, H, W = data.shape

                data = data.view(-1, C, H, W)
                target = target.view(-1)

                data, target = data.to(device), target.to(device)

                # print(f'batch steps in VAL: {batch_idx}')
                # with torch.autocast(
                #     device_type="cuda",
                #     dtype=torch.float16
                # ):
                output = backbone_model(data)
                output = output.view(output.size(0), -1)
                output = classifier(output)

                loss = criterion(output, target)
                val_loss_per_batch += loss.item() * data.size(0)

                prediction = torch.argmax(output, dim=1)
                correct_predictions = sum(pred == tar for pred, tar in zip(prediction, target)).item()

                total_correct_predictions += correct_predictions

                # ✅ ADDED: Clear cache after each batch to prevent OOM
                del data, target, output, loss
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        total_loss = val_loss_per_batch / len(valLoader.dataset)
        total_acc = total_correct_predictions * 100 / len(valLoader.dataset)

        return total_loss, total_acc

    def forward(self, trainLoader, valLoader, epochs, output_path):
        # print(self.backbone_model)
        # scaler = torch.amp.GradScaler("cuda")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        backbone_model = self.backbone_model

        # if torch.cuda.device_count() > 1:
        #     print("Using", torch.cuda.device_count(), "GPUs")
        #     backbone_model = nn.DataParallel(backbone_model)

        classifier = self.classifier
        # if torch.cuda.is_available():
        #     backbone_model.cuda()
        #     classifier.cuda()
        backbone_model = backbone_model.to(device)
        classifier = classifier.to(device)
        # scaler = torch.amp.GradScaler("cuda")

        # scaler = torch.amp.GradScaler("cuda")
        optimizer = torch.optim.Adam(
            [
                {"params": backbone_model.parameters()},
                {"params": classifier.parameters()}
            ],
            lr=self.optimizer_config["lr"],
            weight_decay=self.optimizer_config["weight_decay"]
)
        
        save_interval = self.save_interval
        # optimizer = self.optimizer

        train_losses = []
        val_losses = []
        
        train_accuracies = []
        val_accuracies = []

        train_dataset_length = len(trainLoader.dataset)
        num_of_steps = 0

        for epoch in range(epochs):
            num_of_steps += train_dataset_length
            print(f'epoch: {epoch+1}/{epochs}, steps: {num_of_steps}/{train_dataset_length*epochs}')
            backbone_model, classifier, optimizer, train_loss, train_accuracy = self.train_model(trainLoader,
                                                                                                 backbone_model,
                                                                                                 classifier,
                                                                                                 optimizer,
                                                                                                 device)
            train_losses.append(train_loss)
            train_accuracies.append(train_accuracy)
            # print(f'TRAIN IN EPOCHS: total loss type: {type(train_losses)}, total acc type {type(train_accuracies)}')

            val_loss, val_accuracy = self.eval_model(valLoader, backbone_model, classifier, device)
            val_losses.append(val_loss)
            val_accuracies.append(val_accuracy)
            # print(f'EVAL IN EPOCHS: total loss type: {type(val_losses)}, total acc type {type(val_accuracies)}')

            print(f'\ttrain loss: {train_loss:.4f} - train accuracy: {(train_accuracy):.2f}%, val loss: {val_loss:.4f} - val accuracy: {(val_accuracy):.2f}%')

            # ✅ ADDED: Periodic memory cleanup
            if epoch % 5 == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if epochs-epoch <=5:       
                # Save checkpoint in last 5 epochs
                backbone_path = os.path.join(output_path, f'backbone_epoch_{epoch+1}.pth')
                classifier_path = os.path.join(output_path, f'classifier_epoch_{epoch+1}.pth')
                torch.save(backbone_model.state_dict(), backbone_path)
                torch.save(classifier.state_dict(), classifier_path)
                print(f"Checkpoint saved at epoch {epoch+1}")

In [34]:

import pickle
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

videos_path = '/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos'


preprocess = True

with open(annot_train_pkl, 'rb') as train_pkl, open(annot_val_pkl, 'rb') as val_pkl:
    train_data = pickle.load(train_pkl)
    val_data = pickle.load(val_pkl)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # ✅ FIXED: Enable flexible memory allocation

batch_size = 8  # ✅ CRITICAL: batch_size=16 → 192 images/batch causes OOM. Use 8 → 96 images/batch
# ✅ FIXED: Now using collate_fn for proper memory handling
# ✅ OPTIMIZED: float16 in dataloader reduces memory by ~50% more
train_loader = DataLoader(
    VolleyBallPersonDataLevel(videos_path, train_data, preprocess=preprocess, shuffle=False), 
    batch_size=batch_size, 
    num_workers=2,  # ✅ REDUCED: from 4 to 2 to prevent file descriptor leaks
    collate_fn=collate_fn,  # ✅ FIXED: Use proper collation
    pin_memory=True  # ✅ ADDED: Speed up CPU->GPU transfer
)
val_loader = DataLoader(
    VolleyBallPersonDataLevel(videos_path, val_data, preprocess=preprocess, shuffle=False), 
    batch_size=batch_size, 
    num_workers=2,  # ✅ REDUCED: from 4 to 2
    collate_fn=collate_fn,  # ✅ FIXED: Use proper collation
    pin_memory=True
)

print("train samples:", len(train_loader.dataset), "batches:", len(train_loader))
print("val samples:", len(val_loader.dataset), "batches:", len(val_loader))
print("✅ float16 enabled: Images use ~50% less GPU memory")

num_classes = 9
epochs = 40
my_model = PersonLevelModel(num_classes)

optimizer = "Adam"
lr = 1e-4
weight_decay = 3e-3
optim_params = {
    "optimizer": optimizer,
    "lr": lr,
    "weight_decay": weight_decay
}
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)
save_interval = 10
acc = "accuracy"
my_model.set_metrics(optimizer=optim_params, criterion=criterion, accuracy=acc, save_interval=save_interval)

print(my_model.model_summary())

train samples: 19368 batches: 1211
val samples: 12069 batches: 755
backbone model
Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequen

after it took 220 minutes for one epoch, i decided to stop the training and try to check the problem

### Woooooooooooooooooooooow 
49 min for one epoch
My model is Woooooooooorking

In [36]:
my_model.forward(train_loader, val_loader, epochs, output_path=training_output_path)

epoch: 1/40, steps: 19368/774720


OutOfMemoryError: CUDA out of memory. Tried to allocate 74.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 12.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.30 GiB is allocated by PyTorch, and 113.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### i started the model training in 10:00 PM
### it Taked 30 minutes to train the first baseline for 48 epochs 
#### there was a mistake in the above try "i forgot a condition to stop the eval on step 3"

### Started second training try in 11:00 PM for 100 epochs
### It Taked about 2 HOURS 


### I Started the third try in tuesday at 7:00 PM 

### I started for 80 epochs in 10:25 AM
### Ended in ........

In [ ]:
ty = "/a/b/c/d/f"
print(ty.replace("/a/b/", ""))

In [ ]:
# backbone_model_state_path = training_output_path+'/backbone_model_state_dict.pth'
# classifier_model_state_path = training_output_path+'/classifier_state_dict.pth'

# test_model = ImageLevelModel(num_classes)
# with open(backbone_model_state_path, 'rb') as backnone, open(classifier_model_state_path, 'rb') as classifier:
#     test_model.backbone_model.load_state_dict(torch.load(backnone))
#     test_model.classifier.load_state_dict(torch.load(classifier))

# test_path = root_videos_dataset
# test_loader = DataLoader(VolleyBallDataSet(test_path, train_annot, preprocess=preprocess), batch_size=batch_size)

# test_acc = test_model.test_model(test_loader)
# print(test_acc)